Este primeiro trecho do notebook será responsável por realizar a importações dos dados públicos, os transformando em dataframes interpretáveis pelo SQL. Nesse momento, será utilizado Spark e algumas bibliotecas Python para criar os dataframes.
Devido ao bloqueio da plataforma á conexão direta do JSON, os arquivos com os dados foram importados manualmente via CSV, viabilizando a análise

In [0]:
import pandas as pd # dataframe
from pyspark.sql import SparkSession # processamento

path = "/Workspace/Users/gabriel.sant05@outlook.com/case ipca/dados-brutos-IPCA.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .load(path)

df.createOrReplaceTempView("ipca_ingest")



In [0]:
%sql
drop table if exists ipca_tratado;
CREATE OR REPLACE TEMP VIEW ipca_tratado AS
SELECT
  TO_DATE(data, 'dd/MM/yyyy') AS data_ipca,
  CAST(REPLACE(valor, ',', '.') AS DOUBLE) AS valor_mensal,
  YEAR(TO_DATE(data, 'dd/MM/yyyy')) AS ano,
  MONTH(TO_DATE(data, 'dd/MM/yyyy')) AS mes
FROM ipca_ingest;

SELECT * FROM ipca_tratado;

data_ipca,valor_mensal,ano,mes
1991-01-01,24.66,1991,1
1991-02-01,17.13,1991,2
1991-03-01,24.79,1991,3
1991-04-01,4.16,1991,4
1991-05-01,9.73,1991,5
1991-06-01,12.47,1991,6
1991-07-01,12.42,1991,7
1991-08-01,15.79,1991,8
1991-09-01,15.79,1991,9
1991-10-01,18.02,1991,10


In [0]:
%sql
drop table if exists ipca_anual;
CREATE OR REPLACE TEMP VIEW ipca_anual AS
SELECT
  ano,
  ROUND((EXP(SUM(LN(1 + valor_mensal / 100))) - 1) * 100, 2) AS ipca_acumulado_anual
FROM ipca_tratado
GROUP BY ano
ORDER BY ano;

SELECT * FROM ipca_anual;

ano,ipca_acumulado_anual
1991,492.96
1992,1029.31
1993,2316.2
1994,1040.18
1995,47.22
1996,15.58
1997,5.25
1998,1.22
1999,1.33
2000,3.28


In [0]:
%sql
drop table if exists media_ipca;
CREATE OR REPLACE TEMP VIEW media_ipca AS
SELECT
  ROUND(AVG(ipca_acumulado_anual), 2) AS media_ipca_5anos
FROM ipca_anual
WHERE ano IN (2018, 2019, 2022, 2023, 2024);

SELECT * FROM media_ipca;

media_ipca_5anos
5.08


In [0]:
%sql
CREATE OR REPLACE TABLE ipca_mensal
USING DELTA AS
SELECT * FROM ipca_tratado;

CREATE OR REPLACE TABLE ipca_anual_delta
USING DELTA AS
SELECT * FROM ipca_anual;

CREATE OR REPLACE TABLE kpi_ipca
USING DELTA AS
SELECT * FROM kpi_media_ipca;




num_affected_rows,num_inserted_rows


In [0]:

# Tabela anual 
df_anual = spark.sql("SELECT * FROM ipca_anual_delta").toPandas()
display(spark.createDataFrame(df_anual))


ano,ipca_acumulado_anual
1991,492.96
1992,1029.31
1993,2316.2
1994,1040.18
1995,47.22
1996,15.58
1997,5.25
1998,1.22
1999,1.33
2000,3.28


In [0]:
# Tabela mensal
df_mensal = spark.sql("SELECT * FROM ipca_mensal").toPandas()
display(spark.createDataFrame(df_mensal))

data_ipca,valor_mensal,ano,mes
1991-01-01,24.66,1991,1
1991-02-01,17.13,1991,2
1991-03-01,24.79,1991,3
1991-04-01,4.16,1991,4
1991-05-01,9.73,1991,5
1991-06-01,12.47,1991,6
1991-07-01,12.42,1991,7
1991-08-01,15.79,1991,8
1991-09-01,15.79,1991,9
1991-10-01,18.02,1991,10


In [0]:
# KPI
df_kpi = spark.sql("SELECT * FROM kpi_ipca").toPandas()
display(spark.createDataFrame(df_kpi))

media_ipca_5anos
5.08


In [0]:
%sql
SELECT ano, ROUND(ipca_acumulado_anual / 100, 4) AS ipca_acumulado_anual
FROM ipca_anual_delta
ORDER BY ano

ano,ipca_acumulado_anual
1991,4.9296
1992,10.2931
1993,23.162
1994,10.4018
1995,0.4722
1996,0.1558
1997,0.0525
1998,0.0122
1999,0.0133
2000,0.0328
